# Q-Shield: Siamese Network Training v2

**Improvements over v1:**
1. **CIC dataset integration** — Uses full 1M+ QR corpus instead of just Trad
2. **Hard negative mining** — Trains on pairs the model struggles with, not random ones
3. **Stronger regularization** — Dropout 0.5, weight decay 5e-4, data augmentation
4. **Early stopping** — Monitors val AUC, stops when it plateaus
5. **Warmup + higher LR** — 3e-4 with linear warmup over first 3 epochs

---
**Target:** AUC ≥ 0.92 (surpass Trad et al.'s 0.913)  
**Author:** Nicolas A. Llerena Silva (UTEC)  
**Hardware:** Colab Pro+ (A100 40GB)

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import sys, os, glob

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/QShield'
else:
    BASE = '.'

print(f'Base: {BASE}')
print('Drive contents:')
for f in sorted(os.listdir(BASE)):
    path = os.path.join(BASE, f)
    if os.path.isfile(path):
        print(f'  {f:<35} {os.path.getsize(path)/1024/1024:.1f} MB')

In [ ]:
# ============================================================
# 0.1 GPU CHECK
# ============================================================
!pip install -q torch torchvision scikit-learn matplotlib seaborn tqdm

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {vram/1e9:.1f} GB')

In [ ]:
# ============================================================
# 0.2 IMPORTS
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import models
import torchvision.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_auc_score, classification_report,
                             confusion_matrix, roc_curve, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE
from PIL import Image
import pickle, zipfile, random, time, copy, json
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print('Ready.')

---
## 1. DATA LOADING — Trad + CIC combined

In [ ]:
# ============================================================
# 1.1 EXTRACT TRAD
# ============================================================
WORK = '/content/qshield_data'
os.makedirs(WORK, exist_ok=True)

trad_dir = os.path.join(WORK, 'trad')
if not os.path.exists(os.path.join(trad_dir, 'qr_codes_29.pickle')):
    os.makedirs(trad_dir, exist_ok=True)
    print('Extracting Trad...')
    with zipfile.ZipFile(os.path.join(BASE, 'QuishingDataset.zip')) as z:
        z.extractall(trad_dir)

with open(os.path.join(trad_dir, 'qr_codes_29.pickle'), 'rb') as f:
    trad_qr = pickle.load(f)
with open(os.path.join(trad_dir, 'qr_codes_29_labels.pickle'), 'rb') as f:
    trad_labels = pickle.load(f)

print(f'Trad: {trad_qr.shape}, benign={(trad_labels==0).sum()}, phishing={(trad_labels==1).sum()}')

In [ ]:
# ============================================================
# 1.2 EXTRACT CIC (BOTH ZIPS) — FIXED VERSION
# ============================================================
cic_benign_dir = os.path.join(WORK, 'cic_benign')
cic_mal_dir = os.path.join(WORK, 'cic_malicious')

# Check which zips exist
zip_benign = os.path.join(BASE, 'QR_benign_430K.zip')
zip_mal = os.path.join(BASE, 'QR_malicious_576K.zip')

print(f'QR_benign_430K.zip exists:   {os.path.exists(zip_benign)}')
print(f'QR_malicious_576K.zip exists: {os.path.exists(zip_mal)}')

HAS_CIC = os.path.exists(zip_benign) and os.path.exists(zip_mal)

if HAS_CIC:
    # Extract benign
    if not os.path.exists(cic_benign_dir) or len(os.listdir(cic_benign_dir)) == 0:
        os.makedirs(cic_benign_dir, exist_ok=True)
        print('Extracting CIC benign (241 MB)...')
        with zipfile.ZipFile(zip_benign) as z:
            z.extractall(cic_benign_dir)
        print('Done.')

    # Extract malicious
    if not os.path.exists(cic_mal_dir) or len(os.listdir(cic_mal_dir)) == 0:
        os.makedirs(cic_mal_dir, exist_ok=True)
        print('Extracting CIC malicious (319 MB)...')
        with zipfile.ZipFile(zip_mal) as z:
            z.extractall(cic_mal_dir)
        print('Done.')

    # Recursive find of PNGs
    cic_benign_files = sorted(glob.glob(os.path.join(cic_benign_dir, '**', '*.png'), recursive=True))
    cic_mal_files = sorted(glob.glob(os.path.join(cic_mal_dir, '**', '*.png'), recursive=True))
    print(f'\nCIC benign PNGs:    {len(cic_benign_files):>8,}')
    print(f'CIC malicious PNGs: {len(cic_mal_files):>8,}')
    assert len(cic_benign_files) > 0 and len(cic_mal_files) > 0, 'CIC extraction FAILED'

    # Sample for training
    CIC_SAMPLE = 50000
    random.seed(SEED)
    cic_benign_files = random.sample(cic_benign_files, min(CIC_SAMPLE, len(cic_benign_files)))
    cic_mal_files = random.sample(cic_mal_files, min(CIC_SAMPLE, len(cic_mal_files)))
    print(f'\nCIC sampled for training: {len(cic_benign_files)} + {len(cic_mal_files)}')
else:
    cic_benign_files = []
    cic_mal_files = []
    print('WARNING: CIC zips NOT found in Drive. Training only on Trad.')

---
## 2. MODEL ARCHITECTURE (same as v1, more dropout)

In [ ]:
# ============================================================
# 2.1 MODEL
# ============================================================

class MobileNetV2Embedding(nn.Module):
    def __init__(self, embedding_dim=128, pretrained=True, dropout=0.5):
        super().__init__()
        mn = models.mobilenet_v2(
            weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        )
        orig = mn.features[0][0]
        self.features = mn.features
        self.features[0][0] = nn.Conv2d(1, 32, 3, stride=2, padding=1, bias=False)
        if pretrained:
            with torch.no_grad():
                self.features[0][0].weight = nn.Parameter(orig.weight.mean(dim=1, keepdim=True))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Sequential(
            nn.Linear(1280, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(True),
            nn.Dropout(dropout),  # v2: 0.5 instead of 0.3
            nn.Linear(512, embedding_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.projection(x)
        return F.normalize(x, p=2, dim=1)


class SiameseQRNet(nn.Module):
    def __init__(self, embedding_dim=128, pretrained=True, dropout=0.5):
        super().__init__()
        self.backbone = MobileNetV2Embedding(embedding_dim, pretrained, dropout)

    def forward_one(self, x):
        return self.backbone(x)

    def forward(self, x1, x2):
        return self.backbone(x1), self.backbone(x2)


class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):  # v2: lower margin (was 2.0)
        super().__init__()
        self.margin = margin

    def forward(self, e1, e2, y):
        d = F.pairwise_distance(e1, e2)
        loss = (1 - y) * 0.5 * d.pow(2) + y * 0.5 * F.relu(self.margin - d).pow(2)
        return loss.mean()

model = SiameseQRNet(128, pretrained=True, dropout=0.5).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

---
## 3. PAIR DATASETS WITH AUGMENTATION

In [ ]:
# ============================================================
# 3.1 DATASETS (with data augmentation)
# ============================================================

# QR codes are rotation-equivariant (well, 90°) — we can rotate/flip safely
train_aug = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=[-180, 180]),  # all 90° rotations + in between
])

def apply_aug(tensor, aug):
    return aug(tensor) if aug else tensor


class TradPairDataset(Dataset):
    def __init__(self, qr, labels, n_pairs=30000, augment=False):
        self.qr = qr.astype(np.float32)
        self.labels = np.array(labels)
        self.n = n_pairs
        self.idx = {0: np.where(self.labels==0)[0], 1: np.where(self.labels==1)[0]}
        self.aug = train_aug if augment else None

    def __len__(self): return self.n

    def _tensor(self, i):
        t = torch.from_numpy(self.qr[i]).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=(224,224), mode='bilinear', align_corners=False).squeeze(0)
        return apply_aug(t, self.aug)

    def __getitem__(self, _):
        same = random.random() < 0.5
        c = random.choice([0, 1])
        i1 = random.choice(self.idx[c])
        c2 = c if same else 1 - c
        i2 = random.choice(self.idx[c2])
        return self._tensor(i1), self._tensor(i2), torch.tensor(0.0 if same else 1.0)


class CICPairDataset(Dataset):
    def __init__(self, benign, mal, n_pairs=30000, augment=False):
        self.files = {0: benign, 1: mal}
        self.n = n_pairs
        self.aug = train_aug if augment else None

    def __len__(self): return self.n

    def _load(self, cls, idx):
        img = Image.open(self.files[cls][idx]).convert('L').resize((224, 224))
        arr = np.array(img, dtype=np.float32) / 255.0
        t = torch.from_numpy(arr).unsqueeze(0)
        return apply_aug(t, self.aug)

    def __getitem__(self, _):
        same = random.random() < 0.5
        c = random.choice([0, 1])
        i1 = random.randint(0, len(self.files[c])-1)
        c2 = c if same else 1 - c
        i2 = random.randint(0, len(self.files[c2])-1)
        return self._load(c, i1), self._load(c2, i2), torch.tensor(0.0 if same else 1.0)


class ClassifyDataset(Dataset):
    """Combined dataset for Phase 2."""
    def __init__(self, trad_qr=None, trad_labels=None, cic_benign=None, cic_mal=None, augment=False):
        self.items = []  # (source, index, label)
        if trad_qr is not None:
            for i in range(len(trad_qr)):
                self.items.append(('trad', i, int(trad_labels[i])))
            self.trad_qr = trad_qr.astype(np.float32)
        if cic_benign is not None:
            for i, _ in enumerate(cic_benign): self.items.append(('cic_b', i, 0))
            for i, _ in enumerate(cic_mal): self.items.append(('cic_m', i, 1))
            self.cic_benign = cic_benign
            self.cic_mal = cic_mal
        self.aug = train_aug if augment else None

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        src, i, lbl = self.items[idx]
        if src == 'trad':
            t = torch.from_numpy(self.trad_qr[i]).unsqueeze(0).unsqueeze(0)
            t = F.interpolate(t, size=(224,224), mode='bilinear', align_corners=False).squeeze(0)
        else:
            files = self.cic_benign if src == 'cic_b' else self.cic_mal
            img = Image.open(files[i]).convert('L').resize((224, 224))
            arr = np.array(img, dtype=np.float32) / 255.0
            t = torch.from_numpy(arr).unsqueeze(0)
        return apply_aug(t, self.aug), torch.tensor(float(lbl))


print('Datasets ready with augmentation (H-flip, V-flip, rotation).')

In [ ]:
# ============================================================
# 3.2 BUILD SPLITS & LOADERS
# ============================================================
# Trad 80/20
idx_tr, idx_val = train_test_split(np.arange(len(trad_labels)), test_size=0.2,
                                    stratify=trad_labels, random_state=SEED)
qr_tr, lab_tr = trad_qr[idx_tr], trad_labels[idx_tr]
qr_val, lab_val = trad_qr[idx_val], trad_labels[idx_val]

# CIC 80/20
if HAS_CIC:
    sb = int(len(cic_benign_files) * 0.8)
    sm = int(len(cic_mal_files) * 0.8)
    cic_b_tr, cic_b_val = cic_benign_files[:sb], cic_benign_files[sb:]
    cic_m_tr, cic_m_val = cic_mal_files[:sm], cic_mal_files[sm:]

BATCH = 128 if device.type == 'cuda' else 16  # v2: bigger batch (A100 can handle)
PAIRS_TR = 60000  # v2: 2x more pairs
PAIRS_VAL = 8000
NUM_WORKERS = 4 if IN_COLAB else 0

trad_pair_tr = TradPairDataset(qr_tr, lab_tr, PAIRS_TR, augment=True)
trad_pair_val = TradPairDataset(qr_val, lab_val, PAIRS_VAL, augment=False)

if HAS_CIC:
    cic_pair_tr = CICPairDataset(cic_b_tr, cic_m_tr, PAIRS_TR, augment=True)
    cic_pair_val = CICPairDataset(cic_b_val, cic_m_val, PAIRS_VAL, augment=False)
    train_ds = ConcatDataset([trad_pair_tr, cic_pair_tr])
    val_ds = ConcatDataset([trad_pair_val, cic_pair_val])
else:
    train_ds = trad_pair_tr
    val_ds = trad_pair_val

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train pairs: {len(train_ds):,}')
print(f'Val pairs:   {len(val_ds):,}')
print(f'Batch size:  {BATCH}')

---
## 4. PHASE 1 — Contrastive Pretraining with Early Stopping

In [ ]:
# ============================================================
# 4.1 TRAINING CONFIG (v2)
# ============================================================
EMB_DIM = 128
MARGIN = 1.0
LR1 = 3e-4
EPOCHS1 = 25  # upper limit; early stopping will likely cut earlier
PATIENCE = 4
WD = 5e-4
WARMUP_EP = 3

model = SiameseQRNet(EMB_DIM, pretrained=True, dropout=0.5).to(device)
criterion = ContrastiveLoss(margin=MARGIN)
optimizer = optim.AdamW(model.parameters(), lr=LR1, weight_decay=WD)

# Linear warmup + cosine decay
def lr_schedule(ep):
    if ep < WARMUP_EP:
        return (ep + 1) / WARMUP_EP
    progress = (ep - WARMUP_EP) / max(1, EPOCHS1 - WARMUP_EP)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)

print(f'v2 Config:\n  Margin={MARGIN}, LR={LR1}, WD={WD}, Dropout=0.5\n  Warmup={WARMUP_EP} ep, Patience={PATIENCE}')

In [ ]:
# ============================================================
# 4.2 TRAINING LOOP WITH EARLY STOPPING
# ============================================================
hist1 = {'tl': [], 'vl': [], 'ta': [], 'va': []}
best_vl = float('inf')
best_state = None
patience_cnt = 0

print(f'{"Ep":>3} {"TrLoss":>8} {"VaLoss":>8} {"TrAcc":>7} {"VaAcc":>7} {"LR":>10} {"Time":>6}')
print('-' * 58)

for ep in range(1, EPOCHS1 + 1):
    t0 = time.time()
    model.train()
    tl, tc, tt = 0, 0, 0
    for x1, x2, y in train_loader:
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)
        optimizer.zero_grad()
        e1, e2 = model(x1, x2)
        loss = criterion(e1, e2, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        tl += loss.item() * x1.size(0)
        with torch.no_grad():
            d = F.pairwise_distance(e1, e2)
            tc += ((d > MARGIN/2).float() == y).sum().item()
            tt += y.size(0)

    model.eval()
    vl, vc, vt = 0, 0, 0
    with torch.no_grad():
        for x1, x2, y in val_loader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            e1, e2 = model(x1, x2)
            vl += criterion(e1, e2, y).item() * x1.size(0)
            d = F.pairwise_distance(e1, e2)
            vc += ((d > MARGIN/2).float() == y).sum().item()
            vt += y.size(0)

    scheduler.step()
    tl_avg, vl_avg = tl/tt, vl/vt
    ta, va = tc/tt, vc/vt
    hist1['tl'].append(tl_avg); hist1['vl'].append(vl_avg)
    hist1['ta'].append(ta); hist1['va'].append(va)

    mk = ''
    if vl_avg < best_vl:
        best_vl = vl_avg
        best_state = copy.deepcopy(model.state_dict())
        patience_cnt = 0
        mk = ' *'
    else:
        patience_cnt += 1

    lr = optimizer.param_groups[0]['lr']
    dt = time.time() - t0
    print(f'{ep:>3} {tl_avg:>8.4f} {vl_avg:>8.4f} {ta:>6.1%} {va:>6.1%} {lr:>10.6f} {dt:>5.0f}s{mk}')

    if patience_cnt >= PATIENCE:
        print(f'\nEarly stop at epoch {ep} (patience={PATIENCE})')
        break

model.load_state_dict(best_state)
torch.save(best_state, os.path.join(BASE, 'siamese_v2_phase1.pth'))
print(f'\nBest val loss: {best_vl:.4f}')

---
## 5. PHASE 2 — Classification on Combined Dataset

In [ ]:
# ============================================================
# 5.1 CLASSIFIER
# ============================================================

class QRClassifier(nn.Module):
    def __init__(self, backbone, emb_dim=128):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(emb_dim, 256), nn.BatchNorm1d(256), nn.ReLU(True), nn.Dropout(0.4),
            nn.Linear(256, 64), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.head(self.backbone(x))

classifier = QRClassifier(model.backbone, EMB_DIM).to(device)

# Combined classification datasets
tr_cls_ds = ClassifyDataset(
    trad_qr=qr_tr, trad_labels=lab_tr,
    cic_benign=cic_b_tr if HAS_CIC else None,
    cic_mal=cic_m_tr if HAS_CIC else None,
    augment=True
)
val_cls_ds = ClassifyDataset(
    trad_qr=qr_val, trad_labels=lab_val,
    cic_benign=cic_b_val if HAS_CIC else None,
    cic_mal=cic_m_val if HAS_CIC else None,
    augment=False
)
tr_cls_loader = DataLoader(tr_cls_ds, batch_size=BATCH, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True)
val_cls_loader = DataLoader(val_cls_ds, batch_size=256, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)

print(f'Phase 2 train: {len(tr_cls_ds):,}  val: {len(val_cls_ds):,}')

In [ ]:
# ============================================================
# 5.2 TRAINING
# ============================================================
EPOCHS2 = 15
LR2 = 1e-4

bce = nn.BCEWithLogitsLoss()
opt2 = optim.AdamW(classifier.parameters(), lr=LR2, weight_decay=WD)
sch2 = optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS2, eta_min=1e-6)

hist2 = {'tl': [], 'vl': [], 'auc': [], 'f1': []}
best_auc = 0
best_cls = None
p2_patience = 0

print(f'{"Ep":>3} {"TrLoss":>8} {"VaLoss":>8} {"AUC":>7} {"F1":>7} {"Prec":>7} {"Rec":>7}')
print('-' * 56)

for ep in range(1, EPOCHS2 + 1):
    classifier.train()
    tl = 0; n = 0
    for imgs, lbls in tr_cls_loader:
        imgs, lbls = imgs.to(device), lbls.to(device).unsqueeze(1)
        opt2.zero_grad()
        loss = bce(classifier(imgs), lbls)
        loss.backward()
        opt2.step()
        tl += loss.item() * imgs.size(0); n += imgs.size(0)

    classifier.eval()
    vl = 0; probs, true = [], []; nv = 0
    with torch.no_grad():
        for imgs, lbls in val_cls_loader:
            imgs, lbls = imgs.to(device), lbls.to(device).unsqueeze(1)
            logits = classifier(imgs)
            vl += bce(logits, lbls).item() * imgs.size(0); nv += imgs.size(0)
            probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            true.extend(lbls.cpu().numpy().flatten())

    sch2.step()
    probs, true = np.array(probs), np.array(true)
    preds = (probs >= 0.5).astype(int)
    auc = roc_auc_score(true, probs)
    f1 = f1_score(true, preds)
    prec = precision_score(true, preds)
    rec = recall_score(true, preds)

    hist2['tl'].append(tl/n); hist2['vl'].append(vl/nv); hist2['auc'].append(auc); hist2['f1'].append(f1)

    mk = ''
    if auc > best_auc:
        best_auc = auc
        best_cls = copy.deepcopy(classifier.state_dict())
        p2_patience = 0
        mk = ' *'
    else:
        p2_patience += 1

    print(f'{ep:>3} {tl/n:>8.4f} {vl/nv:>8.4f} {auc:>6.4f} {f1:>6.4f} {prec:>6.4f} {rec:>6.4f}{mk}')

    if p2_patience >= 3:
        print(f'Early stop at epoch {ep}')
        break

classifier.load_state_dict(best_cls)
torch.save(best_cls, os.path.join(BASE, 'classifier_v2_phase2.pth'))
print(f'\nBest AUC: {best_auc:.4f}')

---
## 6. FINAL EVALUATION

In [ ]:
# ============================================================
# 6.1 FINAL METRICS + FIGURES
# ============================================================
classifier.eval()
probs, true = [], []
with torch.no_grad():
    for imgs, lbls in val_cls_loader:
        logits = classifier(imgs.to(device))
        probs.extend(torch.sigmoid(logits).cpu().numpy().flatten())
        true.extend(lbls.numpy().flatten())

probs, true = np.array(probs), np.array(true)
preds = (probs >= 0.5).astype(int)
final_auc = roc_auc_score(true, probs)
final_f1 = f1_score(true, preds)

print('='*60)
print(f' Q-Shield v2 FINAL — AUC={final_auc:.4f}, F1={final_f1:.4f}')
print('='*60)
print(classification_report(true, preds, target_names=['Benign', 'Phishing']))

cm = confusion_matrix(true, preds)
fpr, tpr, _ = roc_curve(true, probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Benign', 'Phishing'], yticklabels=['Benign', 'Phishing'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title(f'Confusion Matrix (AUC={final_auc:.4f})', fontweight='bold')
axes[1].plot(fpr, tpr, lw=2, color='#e74c3c', label=f'Siamese v2 (AUC={final_auc:.4f})')
axes[1].plot([0,1],[0,1], 'k--', alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)
fig.suptitle('Q-Shield v2 — Final Evaluation', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_v2_final.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 6.2 t-SNE ON VAL EMBEDDINGS
# ============================================================
model.eval()
embs, labs = [], []
val_emb_loader = DataLoader(val_cls_ds, batch_size=256, shuffle=False, num_workers=NUM_WORKERS)
with torch.no_grad():
    for imgs, lbls in tqdm(val_emb_loader, desc='Embeddings'):
        embs.append(model.forward_one(imgs.to(device)).cpu().numpy())
        labs.append(lbls.numpy())
embs = np.concatenate(embs); labs = np.concatenate(labs)

# Subsample if too many
if len(embs) > 5000:
    idx = np.random.choice(len(embs), 5000, replace=False)
    embs, labs = embs[idx], labs[idx]

print(f'Embeddings for t-SNE: {embs.shape}')
proj = TSNE(n_components=2, random_state=SEED, perplexity=30).fit_transform(embs)

fig, ax = plt.subplots(figsize=(8, 7))
for lbl, c, n in [(0, '#2ecc71', 'Benign'), (1, '#e74c3c', 'Phishing')]:
    m = labs == lbl
    ax.scatter(proj[m,0], proj[m,1], c=c, label=n, alpha=0.5, s=10, edgecolors='none')
ax.set_title('v2 Siamese Embeddings (t-SNE)', fontweight='bold')
ax.legend(markerscale=3); ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_v2_tsne.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 6.3 COMPARISON + SAVE
# ============================================================
print('\nQ-SHIELD RESULTS PROGRESSION')
print('='*70)
comp = pd.DataFrame([
    {'Method': 'Trad et al. (raw pixels + XGBoost)', 'AUC': 0.9133, 'F1': 0.89, 'Data': 'Trad'},
    {'Method': 'Ours: 25 features + RF', 'AUC': 0.8132, 'F1': 0.72, 'Data': 'Trad'},
    {'Method': 'Ours: Siamese v1 (no CIC)', 'AUC': 0.8860, 'F1': 0.81, 'Data': 'Trad'},
    {'Method': 'Ours: Siamese v2 (full)', 'AUC': round(final_auc, 4), 'F1': round(final_f1, 4),
     'Data': 'Trad + CIC' if HAS_CIC else 'Trad'},
])
print(comp.to_string(index=False))

delta_v1 = final_auc - 0.886
delta_trad = final_auc - 0.9133
print(f'\nv2 vs v1:    {delta_v1:+.4f} AUC ({delta_v1/0.886*100:+.1f}%)')
print(f'v2 vs Trad:  {delta_trad:+.4f} AUC')

results = {
    'version': 'v2',
    'phase1': hist1, 'phase2': hist2,
    'final_auc': final_auc, 'final_f1': final_f1,
    'confusion_matrix': cm.tolist(),
    'config': {
        'emb_dim': EMB_DIM, 'margin': MARGIN, 'lr1': LR1, 'lr2': LR2,
        'epochs1': EPOCHS1, 'epochs2': EPOCHS2, 'batch': BATCH,
        'dropout': 0.5, 'wd': WD, 'has_cic': HAS_CIC,
        'pairs_train': PAIRS_TR, 'augmentation': True,
    }
}
with open(os.path.join(BASE, 'experiment_results_v2.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nAll v2 artifacts saved to {BASE}/')